In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import current_timestamp

target = DeltaTable.forName(spark, "gold.dim_policy")

(
  target.alias("t")
    .merge(
      source_df.alias("s"),
      "t.policy_id = s.policy_id"
    )
    .whenMatchedUpdate(set={
      "customer_name": "s.customer_name",
      "policy_status": "s.policy_status",
      "premium_amount": "s.premium_amount",
      "last_updated_ts": "current_timestamp()"
    })
    .whenNotMatchedInsert(values={
      "policy_id": "s.policy_id",
      "customer_name": "s.customer_name",
      "policy_status": "s.policy_status",
      "premium_amount": "s.premium_amount",
      "last_updated_ts": "current_timestamp()"
    })
    .execute()
)

In [0]:
#Best Example for SCD type 2
target.alias("t").merge(
source.alias("s"),
"t.policy_id = s.policy_id AND t.is_current = true"
).whenMatchedUpdate(
set={"is_current":"false","end_date":"current_date()"}
).whenNotMatchedInsertAll()
.execute()

| Type | What happens on change? | History kept? | Typical columns |
|------|--------------------------|--------------|----------------|
| **SCD1** | Overwrite row | ❌ No | just attributes + audit ts |
| **SCD2** | Expire old row + insert new row | ✅ Full | `is_current`, `start_date`, `end_date` |
| **SCD3** | Shift current → previous + overwrite current | ✅ Limited (1 step) | `attr_current`, `attr_previous` |

In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(spark, "gold.dim_policy_type3")

(
  target.alias("t")
    .merge(
      source_df.alias("s"),
      "t.policy_id = s.policy_id"
    )
    # Update ONLY if the tracked attribute changed
    .whenMatchedUpdate(
      condition="t.status_current <> s.policy_status OR (t.status_current IS NULL AND s.policy_status IS NOT NULL) OR (t.status_current IS NOT NULL AND s.policy_status IS NULL)",
      set={
        "status_previous": "t.status_current",
        "status_current": "s.policy_status",
        "status_change_ts": "current_timestamp()",
        "last_updated_ts": "current_timestamp()"
      }
    )
    # Optional: if match but no change, just touch last_updated_ts (often skipped)
    # .whenMatchedUpdate(
    #   condition="NOT (t.status_current <> s.policy_status OR (t.status_current IS NULL AND s.policy_status IS NOT NULL) OR (t.status_current IS NOT NULL AND s.policy_status IS NULL))",
    #   set={"last_updated_ts": "current_timestamp()"}
    # )
    .whenNotMatchedInsert(values={
      "policy_id": "s.policy_id",
      "status_previous": "NULL",
      "status_current": "s.policy_status",
      "status_change_ts": "current_timestamp()",
      "last_updated_ts": "current_timestamp()"
    })
    .execute()
)

Type 2, we can use a flag as Active_Status "F" or Start date and end date. If end date is null, then its active, if not null then its inactive.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

# 1. Declare the target dimension table (streaming) for customers.
dp.create_streaming_table(
    name = "customer_dimension",
    comment = "Customer dimension table (SCD Type 2)"
)

# 2. Define an AUTO CDC flow to apply changes into the dimension table.
dp.create_auto_cdc_flow(
    target = "customer_dimension",
    source = "customer_updates",      # table or view with incoming change events
    keys = ["id"],                   # primary key column(s)
    sequence_by = col("update_time"),# column that orders the changes (e.g. timestamp)
    ignore_null_updates = False,
    apply_as_deletes = expr("operation = 'DELETE'"),  # treat these as delete events
    except_column_list = ["operation", "update_time"],# exclude control columns
    stored_as_scd_type = "2"        # store all changes as SCD Type 2
)


SQL using Auto CDC

In [0]:
%sql
CREATE FLOW flow_name AS
AUTO CDC INTO target_table
FROM stream(cdc_source_table)
KEYS (key_field_like_id)
APPLY AS DELETE WHEN operation_type = "DELETE"
--APPLY AS INSERT WHEN operation_type = "INSERT"
--APPLY AS UPSERT WHEN operation_type = "UPDATE" OR operation_type = "INSERT"
--APPLY AS UPSERT WHEN operation_type
SEQUENCE BY sequence_field
COLUMNS * EXCEPT (operation_type, sequence_field) --to ignore which column to ignore we use EXCEPT
STORED AS SCD TYPE 1;

In [0]:
%sql
--if the pipelines are already using DLT format, we can use 
APPLY CHANGES into --syntax


In [0]:
SQL Server Insurance DB
        ↓
CDC Extraction
        ↓
Azure Data Factory
        ↓
Azure Data Lake Storage
        ↓
Databricks Auto Loader
        ↓
Bronze Tables
        ↓
Data Cleaning
        ↓
Silver Tables
        ↓
Aggregations
        ↓
Gold Tables
        ↓
Power BI Dashboards